# Train, Evaluate, and Track Baseline vs Challenger Models

In [0]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import mlflow
import mlflow.spark
from mlflow.models.signature import infer_signature

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.types import StringType

## Environment & Volume Setup

In [0]:
# Environment & Volume Setup

# Build Unity Catalog Volume for MLflow (temporary)
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.insurance_claim.mlflow_tmp")
tmp_vol_path = "/Volumes/workspace/insurance_claim/mlflow_tmp"

# Set the environment variable for Spark ML caching
os.environ["SPARKML_TEMP_DFS_PATH"] = tmp_vol_path

## ML Data Preparation (Encoding & Vector Assembly)

Goal: Convert string categories to numeric indices and pack features into a vector.

In [0]:
# ML Data Preparation: Encoding & Vector Assembly using Pipeline
print("--- Starting ML Data Preparation ---")

# Load data from Unity Catalog (Adjust the catalog name if it is not 'main')
dataset_path = "workspace.insurance_claim.ml_ready_dataset"
df = spark.table(dataset_path)

# Define columns to exclude from features (Identifiers and Target Label)
# Including IDs in the training phase can lead to data leakage and severe overfitting.
ignore_cols = [
    # ID AND basic Label
    "ClaimID", "BeneID", "Provider", "is_fraud",
    "AttendingPhysician", "OperatingPhysician", "OtherPhysician",
    
    # Collect ClmDiagnosisCode_1, DiagnosisGroupCode
    "ClmDiagnosisCode_2", "ClmDiagnosisCode_3", "ClmDiagnosisCode_4", 
    "ClmDiagnosisCode_5", "ClmDiagnosisCode_6", "ClmDiagnosisCode_7", 
    "ClmDiagnosisCode_8", "ClmDiagnosisCode_9", "ClmDiagnosisCode_10",
    "ClmProcedureCode_1", "ClmProcedureCode_2", "ClmProcedureCode_3", 
    "ClmProcedureCode_4", "ClmProcedureCode_5", "ClmProcedureCode_6",

    "ClaimStartDt", "ClaimEndDt", "AdmissionDt", "DischargeDt", "DOB", "DOD"
]

# Automatically detect and separate categorical and numeric columns based on data types
categorical_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType) and f.name not in ignore_cols]

numeric_cols = [f.name for f in df.schema.fields if not isinstance(f.dataType, StringType) and f.name not in ignore_cols]

print(f"📌 Categorical Features ({len(categorical_cols)}):\n {categorical_cols}")
print(f"\n📌 Numeric Features ({len(numeric_cols)}):\n {numeric_cols}")

## Building the ML Pipeline

In [0]:
# Building the ML Pipeline

# Create a list of output column names
indexed_cat_cols = [f"{c}_indexed" for c in categorical_cols]

# Use inputCols (plural) and outputCols (plural)
# This creates ONLY 1 model in memory instead of multiple models
indexer = StringIndexer(
    inputCols=categorical_cols, 
    outputCols=indexed_cat_cols, 
    handleInvalid="keep"
)

# Vector Assembler
assembler_inputs = numeric_cols + indexed_cat_cols
assembler = VectorAssembler(
    inputCols=assembler_inputs, 
    outputCol="features", 
    handleInvalid="keep"
)

# Build and Run Pipeline
pipeline = Pipeline(stages=[indexer, assembler])
pipeline_model = pipeline.fit(df)
ml_prepared_df = pipeline_model.transform(df)

print("✅ Data Preparation Completed")

ml_prepared_df.select("features", "is_fraud").toPandas()

## Train-Test Split & Class Distribution Check

In [0]:
print("--- Starting Train-Test Split ---")

# Separate data into Train (80%) and Test (20%)
train_data, test_data = ml_prepared_df.randomSplit([0.8, 0.2], seed=42)

print(f"Total records: {ml_prepared_df.count():,}")
print(f"Training set: {train_data.count():,}")
print(f"Testing set: {test_data.count():,}")

In [0]:
# Check balance of data (Class Distribution) in Training Set

print("\n--- Class Distribution in Training Set ---")
train_data.groupBy("is_fraud").count().toPandas()

### Save as Table

In [0]:
train_data.write.mode("overwrite").saveAsTable("workspace.insurance_claim.train_set")
test_data.write.mode("overwrite").saveAsTable("workspace.insurance_claim.test_set")

print("✅ Saved train_set and test_set to Unity Catalog")

### Model Training & MLflow Logging (Integrated with Pipeline & Feature Importance)

In [0]:
# Model Training & MLflow Logging (Integrated with Pipeline & Feature Importance)

with mlflow.start_run(run_name="Challenger_GBTClassifier_With_Signature") as run:
    
    # Save & Log Pipeline Model
    pipeline_path = "/Volumes/workspace/insurance_claim/mlflow_tmp/fraud_pipeline_model"
    pipeline_model.write().overwrite().save(pipeline_path)
    mlflow.spark.log_model(pipeline_model, "fraud_pipeline_model", dfs_tmpdir=tmp_vol_path)
    print(f"✅ Pipeline logged to MLflow and saved to: {pipeline_path}")


    ####### Model Building #######

    print("Training Challenger Model (GBT)...")
    
    # Train Model
    gbt = GBTClassifier(
        featuresCol="features", 
        labelCol="is_fraud", 
        maxIter=20,     
        maxDepth=5,     
        maxBins=15000
    )
    gbt_model = gbt.fit(train_data)
    
    # Evaluate
    evaluator = BinaryClassificationEvaluator(labelCol="is_fraud", metricName="areaUnderROC")
    gbt_predictions = gbt_model.transform(test_data)
    gbt_auc = evaluator.evaluate(gbt_predictions)
    
    # Log parameters and metrics
    mlflow.log_param("model_type", "Gradient Boosted Trees")
    mlflow.log_param("maxIter", 20)
    mlflow.log_param("maxDepth", 5)
    mlflow.log_param("maxBins", 15000)
    mlflow.log_metric("ROC_AUC", gbt_auc)
    
    # Create Model Signature
    signature = infer_signature(
        model_input=train_data.select("features"), 
        model_output=gbt_predictions.select("prediction")
    )
    
    # Log model WITH signature attached
    mlflow.spark.log_model(
        spark_model=gbt_model, 
        artifact_path="challenger_model", 
        signature=signature,
        dfs_tmpdir=tmp_vol_path
    )
    print(f"✅ Challenger ROC AUC: {gbt_auc:.4f}")

    ############################

    # Extract & Save Feature Importances

    print("🔍 Extracting Feature Importance from the model...")
    importances = gbt_model.featureImportances.toArray()
    
    importance_df = pd.DataFrame({
        'Feature': assembler_inputs,  # extract variables from VectorAssembler
        'Importance': importances
    }).sort_values(by='Importance', ascending=False)
    
    spark_importance_df = spark.createDataFrame(importance_df)
    importance_table_path = "workspace.insurance_claim.gbt_feature_importances"
    spark_importance_df.write.mode("overwrite").saveAsTable(importance_table_path)
    print(f"📊 Feature importances saved to: {importance_table_path}")

    # Automately extract run_id for Model Registry
    run_id = run.info.run_id

## Register the Winning Model to Unity Catalog

In [0]:
# Register the Winning Model to Unity Catalog

print("\n--- Registering Model to Unity Catalog ---")

model_name = "workspace.insurance_claim.fraud_gbt_model"
model_uri = f"runs:/{run_id}/challenger_model"

try:
    registered_model = mlflow.register_model(
        model_uri=model_uri,
        name=model_name
    )
    print(f"✅ Successfully registered model: {registered_model.name} (Version: {registered_model.version})")

except Exception as e:
    print(f"❌ Error registering model: {e}")

### Baseline Model (Logistic Regression)

In [0]:
# =============================================================================
# 🟢 RUN 1: Baseline Model (Logistic Regression)
# =============================================================================

with mlflow.start_run(run_name="Baseline_LogisticRegression"):
    print("Training Baseline Model...")
    lr = LogisticRegression(featuresCol="features", labelCol="is_fraud", maxIter=10)
    lr_model = lr.fit(train_data)
    
    # Make predictions and evaluate
    lr_predictions = lr_model.transform(test_data)
    lr_auc = evaluator.evaluate(lr_predictions)
    
    # Log parameters and metrics
    mlflow.log_param("model_type", "Logistic Regression")
    mlflow.log_metric("ROC_AUC", lr_auc)
    
    # add variable: dfs_tmpdir to point at UC Volume
    mlflow.spark.log_model(lr_model, "baseline_model", dfs_tmpdir=tmp_vol_path)
    
    print(f"✅ Baseline ROC AUC: {lr_auc:.4f}")

### Challenger Model (Random Forest)

In [0]:
# =============================================================================
# 🔵 RUN 2/1: Challenger Model (Random Forest)
# =============================================================================

with mlflow.start_run(run_name="Challenger_RandomForest"):
    print("Training Challenger Model...")
    rf = RandomForestClassifier(
    featuresCol="features", 
    labelCol="is_fraud", 
    numTrees=50, 
    maxDepth=5,
    maxBins=15000
    )
    rf_model = rf.fit(train_data)
    
    # Make predictions and evaluate
    rf_predictions = rf_model.transform(test_data)
    rf_auc = evaluator.evaluate(rf_predictions)
    
    # Log parameters and metrics
    mlflow.log_param("model_type", "Random Forest")
    mlflow.log_param("numTrees", 50)
    mlflow.log_param("maxDepth", 5)
    mlflow.log_metric("ROC_AUC", rf_auc)
    
    # add variable: dfs_tmpdir to pint at UC Volume
    mlflow.spark.log_model(rf_model, "challenger_model", dfs_tmpdir=tmp_vol_path)
    
    print(f"✅ Challenger ROC AUC: {rf_auc:.4f}")

In [0]:
# =============================================================================
# 🔵 RUN 2/2: Challenger Model (Random Forest)
# =============================================================================

with mlflow.start_run(run_name="Challenger_RandomForest"):
    print("Training Challenger Model...")
    
    # ADDED: maxBins=15000 to handle high cardinality in categorical features
    rf = RandomForestClassifier(
        featuresCol="features", 
        labelCol="is_fraud", 
        numTrees=50, 
        maxDepth=5,
        maxBins=15000 
    )
    rf_model = rf.fit(train_data)
    
    # Make predictions on the test set and evaluate the model
    rf_predictions = rf_model.transform(test_data)
    rf_auc = evaluator.evaluate(rf_predictions)
    
    # Log model parameters to MLflow for tracking and comparison
    mlflow.log_param("model_type", "Random Forest")
    mlflow.log_param("numTrees", 50)
    mlflow.log_param("maxDepth", 5)
    mlflow.log_param("maxBins", 15000) # ADDED: Good practice to log the modified maxBins
    
    # Log the evaluation metric (ROC AUC)
    mlflow.log_metric("ROC_AUC", rf_auc)
    
    # ADDED: dfs_tmpdir=tmp_vol_path to safely log the model via Unity Catalog Volume 
    # (Bypasses restrictions on Databricks Shared/Serverless clusters)
    mlflow.spark.log_model(rf_model, "challenger_model", dfs_tmpdir=tmp_vol_path)
    
    print(f"✅ Challenger ROC AUC: {rf_auc:.4f}")

### Updated Strategy
Goal: Use Default Random Forest as Baseline, and GBT as Challenger

In [0]:
# =============================================================================
# 🟢 RUN 3: NEW Baseline Model (Default Random Forest)
# =============================================================================
with mlflow.start_run(run_name="Baseline_Default_RandomForest"):
    print("Training Baseline Model (Default RF)...")
    
    # Using mostly default parameters (except maxBins for your specific data)
    rf_default = RandomForestClassifier(
        featuresCol="features", 
        labelCol="is_fraud",
        maxBins=15000 
        # Not setting numTrees or maxDepth here, letting Spark use its defaults
    )
    rf_baseline_model = rf_default.fit(train_data)
    
    # Evaluate
    baseline_predictions = rf_baseline_model.transform(test_data)
    baseline_auc = evaluator.evaluate(baseline_predictions)
    
    # Log parameters and metrics
    mlflow.log_param("model_type", "Random Forest (Default)")
    mlflow.log_param("maxBins", 15000)
    mlflow.log_metric("ROC_AUC", baseline_auc)
    
    # Log model to UC Volume
    mlflow.spark.log_model(rf_baseline_model, "baseline_model", dfs_tmpdir=tmp_vol_path)
    print(f"✅ Baseline ROC AUC: {baseline_auc:.4f}")

In [0]:
# =============================================================================
# 🔵 RUN 4/1: Gradient Boosted Trees
# =============================================================================

with mlflow.start_run(run_name="Challenger_GBTClassifier_With_Signature"):
    print("Training Challenger Model (GBT)...")
    
    # 1. Train Model
    gbt = GBTClassifier(
        featuresCol="features", 
        labelCol="is_fraud", 
        maxIter=20,     
        maxDepth=5,     
        maxBins=15000
    )
    gbt_model = gbt.fit(train_data)
    
    # 2. Evaluate
    gbt_predictions = gbt_model.transform(test_data)
    gbt_auc = evaluator.evaluate(gbt_predictions)
    
    # Log parameters and metrics
    mlflow.log_param("model_type", "Gradient Boosted Trees")
    mlflow.log_param("maxIter", 20)
    mlflow.log_param("maxDepth", 5)
    mlflow.log_param("maxBins", 15000)
    mlflow.log_metric("ROC_AUC", gbt_auc)
    
    # =============================================================================
    # 🌟 NEW: Create Model Signature
    # Tells Unity Catalog the exact schema of inputs and outputs
    # =============================================================================
    signature = infer_signature(
        model_input=train_data.select("features"), 
        model_output=gbt_predictions.select("prediction")
    )
    
    # =============================================================================
    # 🌟 MODIFIED: Log model WITH signature attached
    # =============================================================================
    mlflow.spark.log_model(
        spark_model=gbt_model, 
        artifact_path="challenger_model", 
        signature=signature,
        dfs_tmpdir=tmp_vol_path
    )
    print(f"✅ Challenger ROC AUC: {gbt_auc:.4f}")

### Hyperparameter Tuning for GBT Classifier using CrossValidator

In [0]:
# =============================================================================
# 🚀 RUN 4/2: Hyperparameter Tuning for GBT
# =============================================================================

with mlflow.start_run(run_name="Tuned_GBTClassifier_TVS"):
    print("Starting Memory-Friendly Tuning for GBT...")
    
    # Initialize Base GBT 
    gbt = GBTClassifier(featuresCol="features", labelCol="is_fraud", maxBins=15000)
    
    # Create Parameter Grid (8 combinations)
    paramGrid = ParamGridBuilder() \
        .addGrid(gbt.maxIter, [20, 50]) \
        .addGrid(gbt.maxDepth, [4, 6]) \
        .addGrid(gbt.stepSize, [0.1, 0.05]) \
        .build()
    
    # 🌟 THE FIX: Use TrainValidationSplit instead of CrossValidator
    # This evaluates each parameter combination only ONCE (using 80% train / 20% validation)
    # It trains 8 models total instead of 24, bypassing the 1GB cache limit!
    tvs = TrainValidationSplit(
        estimator=gbt,
        estimatorParamMaps=paramGrid,
        evaluator=evaluator,
        trainRatio=0.8  # 80% of train_data used for training, 20% for validation during tuning
        # Removed parallelism to keep memory footprint minimal
    )
    
    # Train Models (This will be much faster and consume less memory)
    tvs_model = tvs.fit(train_data)
    best_gbt = tvs_model.bestModel
    
    # Evaluate the BEST Model on the Hold-out Test Set
    best_predictions = best_gbt.transform(test_data)
    best_auc = evaluator.evaluate(best_predictions)
    
    # Log parameters of the winning model to MLflow
    mlflow.log_param("model_type", "Tuned GBT (TVS)")
    mlflow.log_param("best_maxIter", best_gbt.getMaxIter())
    mlflow.log_param("best_maxDepth", best_gbt.getMaxDepth())
    mlflow.log_param("best_stepSize", best_gbt.getStepSize())
    mlflow.log_metric("ROC_AUC", best_auc)
    
    # Log the best model safely
    mlflow.spark.log_model(best_gbt, "tuned_gbt_model", dfs_tmpdir=tmp_vol_path)
    
    print(f"✅ Tuned GBT ROC AUC: {best_auc:.4f}")
    print(f"🏆 Winning Parameters: maxIter={best_gbt.getMaxIter()}, maxDepth={best_gbt.getMaxDepth()}, stepSize={best_gbt.getStepSize()}")